# Multi-Agent Sample Size Calculator

This notebook demonstrates the use of a multi-agent system for clinical trial sample size calculation using LangChain and LangGraph.

## System Architecture

The system consists of four specialized agents:
1. **Parameter Validator Agent**: Validates and structures input parameters
2. **Calculation Agent**: Performs statistical calculations (with MCP server integration)
3. **Visualization Agent**: Creates summaries and recommendations
4. **Coordinator Agent**: Assembles final results


In [ ]:
# Import required libraries
import os
import asyncio
import json
from pathlib import Path
import sys

# Add the current directory to Python path to import our module
sys.path.append('.')

# Import our multi-agent system
from SampleSizeMultiAgent import MultiAgentSampleSizeCalculator, SampleSizeParameters

# Load environment variables
import dotenv
dotenv.load_dotenv()

print("✅ Imports completed successfully")

## Configuration

Make sure your `.env` file contains:
- `ILIAD_API_KEY`: Your Azure OpenAI API key
- `ILIAD_URL_BASE`: Your Azure OpenAI endpoint URL

Also ensure your R MCP server is running on port 9290 for full functionality.

In [ ]:
# Initialize the Multi-Agent Sample Size Calculator
print("🚀 Initializing Multi-Agent Sample Size Calculator...")

try:
    calculator = MultiAgentSampleSizeCalculator()
    print("✅ Calculator initialized successfully")
    print(f"   LLM Model: {calculator.llm.model_name if hasattr(calculator.llm, 'model_name') else 'Azure GPT-4o'}")
    print(f"   MCP Client: {'Connected' if calculator.mcp_client else 'Not available (fallback mode)'}")
    print(f"   Workflow: {'Built' if calculator.workflow else 'Failed to build'}")
except Exception as e:
    print(f"❌ Failed to initialize calculator: {e}")

## Example 1: Basic Three-Arm Clinical Trial

This example demonstrates a typical three-arm trial with placebo and two treatment doses, similar to your original R example.

In [ ]:
# Example 1: Three-arm trial input
example1_input = """
Please calculate sample size for a clinical trial with the following design:

Study Design:
- Multiplicity Procedure: Graphical Procedure
- Treatment arms: Placebo, ABBV932L (low dose), ABBV932H (high dose)
- Allocation ratio: 1:1:1 (equal allocation)

Efficacy Endpoints:
- Primary endpoint effect sizes: -0.33 (low dose vs placebo), -0.33 (high dose vs placebo)
- Secondary endpoint effect sizes: -0.33, -0.33, -0.28, -0.28
- Between-endpoints correlation matrix: [1, 0.6, 0.6, 0.6, 1, 0.6, 0.6, 0.6, 1]

Statistical Parameters:
- Required statistical power: 80% (0.8)
- Significance level (alpha): 5% (0.05)
- Expected drop-out rate: 20% (0.2)

Sample Size Search Parameters:
- Minimum sample size per arm: 50
- Maximum sample size per arm: 300
- Sample size increment: 10

Multiplicity Control:
- Initial hypothesis weights: [0.5, 0.5]
- Transition matrix: [0, 1, 1, 0]
- Success criteria: DisjunctivePower

Simulation Settings:
- Number of simulations: 10000
- Random seed: 12345 (for reproducibility)
"""

print("📝 Example 1 input prepared")
print(f"Input length: {len(example1_input)} characters")

In [ ]:
# Run Example 1 analysis
print("🔄 Running Example 1 analysis...")
print("This may take a few minutes depending on MCP server availability and LLM response times.")

try:
    result1 = await calculator.run_analysis(example1_input)
    print("\n" + "="*80)
    print("EXAMPLE 1 RESULTS")
    print("="*80)
    print(result1)
except Exception as e:
    print(f"❌ Example 1 failed: {e}")

## Example 2: Two-Arm Trial with Different Parameters

This example shows a simpler two-arm trial design.

In [ ]:
# Example 2: Two-arm trial input
example2_input = """
Calculate sample size for a two-arm superiority trial:

Study Design:
- Treatment arms: Placebo, Active Treatment
- Allocation ratio: 1:1
- Multiplicity procedure: Graphical Procedure

Efficacy Parameters:
- Primary endpoint effect size: -0.4 (medium effect)
- No secondary endpoints
- Correlation matrix: [1] (single endpoint)

Statistical Requirements:
- Power: 90% (0.9)
- Alpha: 0.05
- Drop-out rate: 15% (0.15)

Sample Size Range:
- Minimum: 30 per arm
- Maximum: 200 per arm  
- Increment: 5

Multiplicity Settings:
- Initial weights: [1.0]
- Transition matrix: [0]
- Success criteria: ConjunctivePower

Simulation:
- Simulations: 5000
- Seed: 54321
"""

print("📝 Example 2 input prepared")

In [ ]:
# Run Example 2 analysis
print("🔄 Running Example 2 analysis...")

try:
    result2 = await calculator.run_analysis(example2_input)
    print("\n" + "="*80)
    print("EXAMPLE 2 RESULTS")
    print("="*80)
    print(result2)
except Exception as e:
    print(f"❌ Example 2 failed: {e}")

## Example 3: Advanced Four-Arm Trial

This example demonstrates a more complex scenario with multiple arms and endpoints.

In [ ]:
# Example 3: Four-arm trial with complex multiplicity
example3_input = """
Design a sample size calculation for a complex four-arm dose-finding study:

Study Arms:
- Placebo
- Low dose (5mg)
- Medium dose (10mg) 
- High dose (20mg)
- Allocation: 1:1:1:1

Endpoints:
- Primary: Efficacy at 12 weeks
  - Effect sizes vs placebo: -0.2, -0.35, -0.5
- Key secondary: Safety score
  - Effect sizes vs placebo: -0.15, -0.25, -0.35
- Correlation between endpoints: 0.4
- Full correlation matrix: [1, 0.4, 0.4, 1]

Statistical Design:
- Power requirement: 85%
- Alpha level: 0.025 (more stringent)
- Expected dropout: 25%

Sample Size Search:
- Range: 40 to 150 per arm
- Step size: 5

Multiplicity Control:
- Graphical procedure with weighted hypotheses
- Initial weights: [0.4, 0.4, 0.2] (prioritize higher doses)
- Transition matrix: [0, 0.5, 0.5, 0.5, 0, 0.5, 0.5, 0.5, 0]
- Success criteria: WeightedPower with weights [0.6, 0.4]

Simulation:
- 15000 simulations for precision
- Seed: 98765
"""

print("📝 Example 3 input prepared (complex four-arm trial)")

In [ ]:
# Run Example 3 analysis
print("🔄 Running Example 3 analysis (this may take longer due to complexity)...")

try:
    result3 = await calculator.run_analysis(example3_input)
    print("\n" + "="*80)
    print("EXAMPLE 3 RESULTS")
    print("="*80)
    print(result3)
except Exception as e:
    print(f"❌ Example 3 failed: {e}")

## Testing Individual Agents

This section demonstrates how to test individual agents in the multi-agent system.

In [ ]:
# Test MCP server connection
print("🔍 Testing MCP Server Connection...")

if calculator.mcp_client:
    try:
        tools = await calculator.get_mcp_tools()
        print(f"✅ MCP server connected successfully")
        print(f"   Available tools: {len(tools)}")
        for i, tool in enumerate(tools):
            tool_name = getattr(tool, 'name', f'Tool_{i}')
            tool_desc = getattr(tool, 'description', 'No description')
            print(f"   - {tool_name}: {tool_desc[:100]}...")
    except Exception as e:
        print(f"⚠️ MCP server connection issue: {e}")
else:
    print("⚠️ MCP client not initialized - using fallback calculations")

In [ ]:
# Test parameter validation directly
print("🔍 Testing Parameter Validation Agent...")

test_state = {
    "messages": [{"content": example1_input}],
    "parameters": None,
    "validation_result": None,
    "errors": []
}

try:
    # This would normally be called as part of the workflow
    # but we can test the logic separately
    print("Parameter validation logic is integrated into the workflow.")
    print("See the full workflow examples above for validation results.")
except Exception as e:
    print(f"❌ Parameter validation test failed: {e}")

## Utility Functions

Helper functions for working with the multi-agent system.

In [ ]:
def create_sample_input(arms, effect_sizes, power=0.8, alpha=0.05):
    """
    Helper function to create standardized input for the calculator
    """
    arm_names = ', '.join(arms)
    es_str = ', '.join(map(str, effect_sizes))
    
    return f"""
Calculate sample size for clinical trial:

Arms: {arm_names}
Allocation: {':'.join(['1'] * len(arms))}
Primary effect sizes: {es_str}
Power: {power}
Alpha: {alpha}
Dropout rate: 0.2
Sample size range: 50 to 300 per arm, step 10
Correlation matrix: [1] (single endpoint)
Initial weights: [1.0]
Transition matrix: [0]
Success criteria: DisjunctivePower
Simulations: 5000
"""

# Example usage
quick_input = create_sample_input(
    arms=["Placebo", "Treatment"], 
    effect_sizes=[-0.3],
    power=0.8,
    alpha=0.05
)

print("🛠️ Utility function created. Example input:")
print(quick_input)

In [ ]:
# Quick test with utility function
print("🚀 Running quick test with utility function...")

try:
    quick_result = await calculator.run_analysis(quick_input)
    print("\n" + "="*60)
    print("QUICK TEST RESULTS")
    print("="*60)
    print(quick_result)
except Exception as e:
    print(f"❌ Quick test failed: {e}")

## Summary

This notebook demonstrates a comprehensive multi-agent system for clinical trial sample size calculation that:

### Key Features:
1. **Multi-Agent Architecture**: Four specialized agents handling different aspects
2. **MCP Integration**: Direct connection to R statistical computing server
3. **Fallback Calculations**: Works even without MCP server
4. **Structured Validation**: Comprehensive parameter validation and error handling
5. **LangGraph Workflow**: Orchestrated using state-of-the-art workflow management

### Agents:
- **Parameter Validator**: Extracts and validates statistical parameters
- **Calculation Agent**: Performs sample size calculations via MCP or fallback
- **Visualization Agent**: Creates summaries and recommendations  
- **Coordinator Agent**: Assembles final results and handles workflow

### Applications:
- Clinical trial design
- Power analysis
- Multiplicity adjustment procedures
- Graphical procedures for multiple testing

The system provides a robust, scalable approach to statistical consulting with AI agents.